Pulls data from API, filters for taxon_id 1280 = Staphylococcus aureus that was identified as resistant via wet lab. Then pull out datapoints that show resistance or susceptibility specifically to methicillin.

In [1]:
import requests
import pandas as pd
from io import StringIO

# The URL to pull Staph aureus, make sure that the resistance was tested in wet lab
url = "https://www.bv-brc.org/api/genome_amr/?eq(taxon_id,1280)&eq(evidence,Laboratory%20Method)&limit(50000)"
headers = {"Accept": "text/csv"}

# API request
response = requests.get(url, headers=headers)
df = pd.read_csv(StringIO(response.text))

# Filter to methicillin, only get the Resistant or susceptible genomes
methicillin_df = df[df['Antibiotic'] == 'methicillin'].copy()
methicillin_df = methicillin_df[methicillin_df['Resistant Phenotype'].isin(['Resistant', 'Susceptible'])]

# Get some information on the dataset that was pulled and filtered
print("Shape:", methicillin_df.shape)
print("\nClass balance:")
print(methicillin_df['Resistant Phenotype'].value_counts())
print("\nSample genome IDs:")
print(methicillin_df['Genome ID'].head(10).tolist())

# Save it
methicillin_df.to_csv("methicillin_labels.csv", index=False)
print("\nSaved methicillin_labels.csv")

Shape: (1082, 21)

Class balance:
Resistant Phenotype
Susceptible    574
Resistant      508
Name: count, dtype: int64

Sample genome IDs:
[1280.4727, 1280.8343, 1280.2483, 1280.8315, 1280.7953, 1280.25081, 1280.24873, 1280.2511, 1280.8168, 1280.9764]

Saved methicillin_labels.csv


In [2]:
# Take a look at data table
methicillin_df.head()

,Taxon ID,Genome ID,Genome Name,Antibiotic,Resistant Phenotype,Measurement,Measurement Sign,Measurement Value,Measurement Unit,Laboratory Typing Method,...,Laboratory Typing Platform,Vendor,Testing Standard,Testing Standard Year,Computational Method,Computational Method Version,Computational Method Performance,Evidence,Source,PubMed
17,1280,1280.4727,Staphylococcus aureus strain USFL197,methicillin,Resistant,NaN,NaN,NaN,NaN,Disk diffusion,...,NaN,NaN,CLSI,2013.0,NaN,NaN,NaN,Laboratory Method,NaN,29487865.0
20,1280,1280.8343,Staphylococcus aureus C00013400,methicillin,Susceptible,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Laboratory Method,NaN,26686880.0
142,1280,1280.2483,Staphylococcus aureus strain 12483_8_66,methicillin,Resistant,NaN,NaN,NaN,NaN,Broth dilution,...,VITEK 2,BioMerieux,British Society for Antimicrobial Chemotherapy...,NaN,NaN,NaN,NaN,Laboratory Method,NaN,30696529.0
143,1280,1280.8315,Staphylococcus aureus C00013372,methicillin,Susceptible,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Laboratory Method,NaN,26686880.0
152,1280,1280.7953,Staphylococcus aureus C00001155,methicillin,Susceptible,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Laboratory Method,NaN,26686880.0


Pull all gene results for the Genome_Ids that we collected

In [63]:
import requests
import pandas as pd
from io import StringIO

# Convert the genome IDs to a flat string that can be fed into an API call
genome_list = methicillin_df['Genome ID'].tolist()
genome_string = ",".join([str(x) for x in genome_list])

# Set up batches since there are a lot of genome IDs
all_batches = []
cursor = "*"

# Loop until all batches are completed
while True:
    # API request done in batches
    response = requests.post(
        "https://www.bv-brc.org/api/sp_gene/",
        headers={"Accept": "text/csv", "Content-Type": "application/rqlquery+x-www-form-urlencoded"},
        data=f'in(genome_id,({genome_string}))&eq(property,%22Antibiotic%20Resistance%22)&limit(25000)&cursor({cursor})'
    )

    # Convert the batch API call into a csv and then make into a DF. Append to final batch df
    batch_df = pd.read_csv(StringIO(response.text))
    all_batches.append(batch_df)

    # Save the next_cursor to know where to start for the next API call
    next_cursor = response.headers.get("X-Cursor-Mark")

    # Stop when cursor stops changing
    if next_cursor == cursor:
        break

    # update the Cursor and run the loop again
    cursor = next_cursor

genes_df = pd.concat(all_batches, ignore_index=True)
print(f"\nTotal records: {len(genes_df)}")
print(f"Unique genomes: {genes_df['Genome ID'].nunique()}")
print(f"\nTop 20 genes:")
print(genes_df['Gene'].value_counts().head(20))
print(f"\nmecA present: {'mecA' in genes_df['Gene'].values}")

Cursor: *... | Records: 25000 | Range: items 0-25000/86803
Cursor: AoE/BTQ5ODNkMjJhLWE3... | Records: 25000 | Range: items 0-25000/86803
Cursor: AoE/BTkzN2E4M2ViLWY3... | Records: 25000 | Range: items 0-25000/86803
Cursor: AoE/BWRkMmU1NjQ4LTM1... | Records: 11803 | Range: items 0-11803/86803
Cursor: AoE/BWZmZmY3NzYxLTA5... | Records: 0 | Range: items 0-0/86803

Total records: 86803
Unique genomes: 980

Top 20 genes:
Gene
GdpD         902
gyrB         650
gyrA         647
rpoC         638
rpoB         638
MurA         600
blaZ         368
mgrA         357
arlS         341
folA, Dfr    339
mecR1        339
pgsA         338
mprF         338
arlR         338
mepA         338
dfrC         324
rho          321
folP         319
gidB         319
parE         319
Name: count, dtype: int64

mecA present: True


In [64]:
# Save raw genes
genes_df.to_csv("raw_genes.csv", index=False)
print("Saved raw_genes.csv")

print(f"\nmecA count: {genes_df['Gene'].value_counts()['mecA']}")
print(f"mecA rank: {list(genes_df['Gene'].value_counts().index).index('mecA') + 1}")

Saved raw_genes.csv

mecA count: 298
mecA rank: 50
